In [1]:
import os, shutil
src = None
for root, dirs, files in os.walk("/kaggle/input"):
    dirs[:] = [d for d in dirs if d != "competitions"]
    if "10_train_student_d32.py" in files:
        src = os.path.join(root, "10_train_student_d32.py"); break
assert src, "10_train_student_d32.py tidak ditemukan; tambahkan input rsna-knee-14-student-d32"
shutil.copy(src, "/kaggle/working/10_train_student_d32.py")
print("Script disalin dari:", src)

Script disalin dari: /kaggle/input/notebooks/kellenvalerie/rsna-knee-14-student-d32/10_train_student_d32.py


In [2]:
%%bash
cd /kaggle/working
COMMON="--alpha 0.5 --lr_backbone 1e-4 --oof oof_effb0_kf2.csv --llrd 0.8 --meta _meta.csv"

# GPU 0: DINOv2-Base, data v1, 8 epoch (batch 1 x accum 4)
CUDA_VISIBLE_DEVICES=0 python -u 10_train_student_d32.py $COMMON \
    --backbone vit_base_patch14_dinov2.lvd142m --batch 1 --accum 4 --epochs 8 \
    --tag dinov2b_student_v1d \
    > log_dinov2b_student_v1d.txt 2>&1 &

# GPU 1: DINOv2-S data v1, seed 7 lalu seed 11
( CUDA_VISIBLE_DEVICES=1 python -u 10_train_student_d32.py $COMMON \
    --backbone vit_small_patch14_dinov2.lvd142m --batch 4 --epochs 12 --seed 7 \
    --tag dinov2s_student_v1d_s7 > log_dinov2s_student_v1d_s7.txt 2>&1
  CUDA_VISIBLE_DEVICES=1 python -u 10_train_student_d32.py $COMMON \
    --backbone vit_small_patch14_dinov2.lvd142m --batch 4 --epochs 12 --seed 11 \
    --tag dinov2s_student_v1d_s11 > log_dinov2s_student_v1d_s11.txt 2>&1 ) &

wait
for t in dinov2b_student_v1d dinov2s_student_v1d_s7 dinov2s_student_v1d_s11; do
  echo "===== $t ====="; tail -n 16 log_$t.txt
done
grep -H "Volume :\|Label  :\|Label diperbaiki\|Train \|Jumlah irisan\|Jumlah view\|LLRD\|parameter\|Error\|error" log_*.txt

===== dinov2b_student_v1d =====
Epoch 7: loss 0.3777 | val AUC 0.8718 | val-ref AUC 0.8897 | gold AUC 0.8530 | 45.9 menit

=== [dinov2b_student_v1d] Checkpoint terbaik (val-ref AUC 0.8897) pada label emas ===
ACL                AUC=0.896
MCL                AUC=0.902
Medial Meniscus    AUC=0.851
Lateral Meniscus   AUC=0.696
Medial OA          AUC=0.981
Lateral OA         AUC=0.824
PF OA              AUC=0.789
Effusion           AUC=0.889
Synovitis          AUC=0.729
Baker's            AUC=0.962
Contusion          AUC=0.836
Fracture           AUC=0.896
Mean AUC gold: 0.8543
===== dinov2s_student_v1d_s7 =====
Epoch 11: loss 0.3721 | val AUC 0.8745 | val-ref AUC 0.8922 | gold AUC 0.8516 | 14.1 menit

=== [dinov2s_student_v1d_s7] Checkpoint terbaik (val-ref AUC 0.8922) pada label emas ===
ACL                AUC=0.885
MCL                AUC=0.884
Medial Meniscus    AUC=0.870
Lateral Meniscus   AUC=0.729
Medial OA          AUC=0.988
Lateral OA         AUC=0.799
PF OA              AUC=0.772
Ef

In [3]:
import pandas as pd
for tag in ["dinov2b_student_v1d", "dinov2s_student_v1d_s7", "dinov2s_student_v1d_s11"]:
    print(f"\n===== {tag} =====")
    try:
        print(pd.read_csv(f"/kaggle/working/train_log_{tag}.csv").round(4).to_string(index=False))
    except FileNotFoundError:
        print("log belum ada")


===== dinov2b_student_v1d =====
 epoch   loss  val_auc  val_ref_auc  gold_auc     min
     0 0.5283   0.7640       0.7797    0.7928 45.5400
     1 0.4651   0.8283       0.8480    0.8315 45.4660
     2 0.4391   0.8409       0.8613    0.8270 45.5626
     3 0.4220   0.8483       0.8676    0.8468 45.7257
     4 0.4076   0.8662       0.8843    0.8534 45.8784
     5 0.3942   0.8687       0.8870    0.8513 45.8857
     6 0.3835   0.8719       0.8897    0.8543 45.9238
     7 0.3777   0.8718       0.8897    0.8530 45.9363

===== dinov2s_student_v1d_s7 =====
 epoch   loss  val_auc  val_ref_auc  gold_auc     min
     0 0.5348   0.7821       0.7956    0.8181 14.0771
     1 0.4667   0.8198       0.8360    0.8419 14.2360
     2 0.4456   0.8338       0.8482    0.8407 14.2676
     3 0.4343   0.8403       0.8569    0.8373 14.1962
     4 0.4240   0.8542       0.8695    0.8415 14.1904
     5 0.4136   0.8643       0.8833    0.8534 14.2384
     6 0.4031   0.8669       0.8846    0.8460 14.1495
     7 0.3942